## Imports


In [18]:
import polars as pl
from pathlib import Path

# Data Path


In [19]:
data_folder = Path("../data")

# Load Dataset


In [20]:
sales_df = pl.read_parquet(
    data_folder / "sales_cleaned.parquet"
)

# Preview Dataset
sales_df.head()

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499


## Create Year-Month Column


In [21]:
# Extract year-month for aggregation
sales_df = sales_df.with_columns(
    pl.col("purchase_date").dt.strftime("%Y-%m").alias("year_month")
)

# Monthly Sales Trends


In [22]:
monthly_sales = sales_df.group_by("year_month").agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue"),
    pl.col("price").mean().round().alias("average_order_value")
).sort("year_month")

monthly_sales

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22324\2184171659.py:2: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


year_month,transactions,total_revenue,average_order_value
str,u32,i64,f64
"""2024-01""",3297,1069903,325.0
"""2024-02""",3230,1051770,326.0
"""2024-03""",6672,2191528,328.0
"""2024-04""",3238,1042062,322.0
"""2024-05""",5068,1643682,324.0
…,…,…,…
"""2024-08""",3421,1115879,326.0
"""2024-09""",3262,1064738,326.0
"""2024-10""",5709,1844991,323.0


# Revenue Share by Month


In [23]:
monthly_sales = monthly_sales.with_columns(
    (
        pl.col("total_revenue") / pl.col("total_revenue").sum() * 100
    ).round().alias("revenue_share_pct")
)

monthly_sales.sort("revenue_share_pct", descending=True)

year_month,transactions,total_revenue,average_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""2024-03""",6672,2191528,328.0,13.0
"""2024-10""",5709,1844991,323.0,11.0
"""2024-05""",5068,1643682,324.0,10.0
"""2024-06""",4970,1617130,325.0,10.0
"""2024-11""",4315,1394035,323.0,9.0
…,…,…,…,…
"""2024-08""",3421,1115879,326.0,7.0
"""2024-09""",3262,1064738,326.0,7.0
"""2024-12""",3439,1111261,323.0,7.0


# Festival Monthly Trends


In [ ]:
monthly_festival_sales = sales_df.group_by(["year_month", "festival"]
                                           ).agg(
    pl.count().alias("transactions"),
    pl.col("revenue").sum().alias("total_revenue")
).sort(["year_month", "total_revenue"], descending=[False, True])

monthly_festival_sales.head(10)

C:\Users\Mudit PC\AppData\Local\Temp\ipykernel_22324\2365063454.py:3: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("transactions"),


year_month,festival,transactions,total_revenue
str,str,u32,i64
"""2024-01""",null,3297,1069903
"""2024-02""",null,3230,1051770
"""2024-03""","""Holi""",3331,1097119
"""2024-03""",null,3341,1094409
"""2024-04""",null,3238,1042062
"""2024-05""",null,3451,1113199
"""2024-05""","""Summer""",1617,530483
"""2024-06""",null,3369,1096131
"""2024-06""","""Summer""",1601,520999


# Adding month column to sales_df


In [25]:
sales_df = sales_df.with_columns(
    pl.col("purchase_date").dt.strftime("%B").alias("month_name")
)

# Save Outputs


In [ ]:
monthly_festival_sales.write_parquet(
    data_folder / "monthly_festival_sales.parquet")
monthly_sales.write_parquet(data_folder / "monthly_sales.parquet")

In [27]:
monthly_sales

year_month,transactions,total_revenue,average_order_value,revenue_share_pct
str,u32,i64,f64,f64
"""2024-01""",3297,1069903,325.0,7.0
"""2024-02""",3230,1051770,326.0,6.0
"""2024-03""",6672,2191528,328.0,13.0
"""2024-04""",3238,1042062,322.0,6.0
"""2024-05""",5068,1643682,324.0,10.0
…,…,…,…,…
"""2024-08""",3421,1115879,326.0,7.0
"""2024-09""",3262,1064738,326.0,7.0
"""2024-10""",5709,1844991,323.0,11.0


In [28]:
monthly_festival_sales

year_month,festival,transactions,total_revenue
str,str,u32,i64
"""2024-01""",null,3297,1069903
"""2024-02""",null,3230,1051770
"""2024-03""","""Holi""",3331,1097119
"""2024-03""",null,3341,1094409
"""2024-04""",null,3238,1042062
…,…,…,…
"""2024-10""",null,3375,1091575
"""2024-10""","""Diwali""",2334,753416
"""2024-11""",null,3285,1059115


In [29]:
sales_df

transaction_id,customer_id,product_id,purchase_date,festival,city,product_name,category,customer_name,age,gender,price,revenue,year_month,month_name
str,str,str,datetime[μs],str,str,str,str,str,i64,str,i64,i64,str,str
"""T82""","""c377""","""p3""",2024-01-01 00:00:00,null,"""Jorhat""","""Lip Balm""","""Makeup""","""Harshil Varty""",21,"""Female""",149,149,"""2024-01""","""January"""
"""T910""","""c202""","""p1""",2024-01-01 00:00:00,null,"""Indore""","""Vitamin C Serum""","""Skincare""","""Tanvi Chana""",51,"""Male""",499,499,"""2024-01""","""January"""
"""T1273""","""c752""","""p2""",2024-01-01 00:00:00,null,"""Secunderabad""","""Herbal Shampoo""","""Haircare""","""Idika Roy""",48,"""Male""",199,199,"""2024-01""","""January"""
"""T1369""","""c739""","""p0""",2024-01-01 00:00:00,null,"""Bhatpara""","""Aloe Vera Gel""","""Skincare""","""Harita Comar""",49,"""Female""",299,299,"""2024-01""","""January"""
"""T2858""","""c169""","""p1""",2024-01-01 00:00:00,null,"""Bhagalpur""","""Vitamin C Serum""","""Skincare""","""Zayyan Mander""",51,"""Female""",499,499,"""2024-01""","""January"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""T48069""","""c243""","""p3""",2024-12-31 00:00:00,null,"""Solapur""","""Lip Balm""","""Makeup""","""Eiravati Pillai""",44,"""Male""",149,149,"""2024-12""","""December"""
"""T48763""","""c716""","""p0""",2024-12-31 00:00:00,null,"""Parbhani""","""Aloe Vera Gel""","""Skincare""","""Falguni Bakshi""",19,"""Female""",299,299,"""2024-12""","""December"""
"""T48997""","""c609""","""p2""",2024-12-31 00:00:00,null,"""Ambala""","""Herbal Shampoo""","""Haircare""","""Samarth Ray""",35,"""Female""",199,199,"""2024-12""","""December"""
